# Извлечение Perception Encoder (PE-Core) фич фото

Прогоняет 144313 фото объявлений через замороженный Meta Perception Encoder PE-Core-L14-336 и сохраняет ДВА выхода:
- сырой image-эмбеддинг (1024d) - ещё один image-блок рядом с DINOv3
- prompt-similarity: косинус к промптам-ловушкам (lot/accessory/parts/digital/movie + типы) - новый сигнал под should_abstain

PE-Core новее и сильнее SigLIP2 на fine-grained/long-tail (ObjectNet, ImageNet-A) - ровно наш кейс (тонкие типы + редкие ловушки)

Вход: папка image, загружена как Kaggle Dataset
Выход: /kaggle/working/image_emb_pe.parquet (image_id + 1024) и text_sim_pe.parquet (image_id + sim_*)

Запуск: Accelerator GPU T4 x2, Internet ON, Run All

In [ ]:
!pip install -q -U open_clip_torch

## Токен HF

PE-Core (Apache-2.0) публичная, open_clip качает с hf-hub без токена. Если упрётся в gated/rate-limit - заведи Kaggle Secret с HF-токеном и раскомментируй login ниже

In [ ]:
# from huggingface_hub import login
# from kaggle_secrets import UserSecretsClient
# login(token=UserSecretsClient().get_secret('HF_TOKEN'))

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import open_clip
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

In [ ]:
MODEL_NAME = 'hf-hub:timm/PE-Core-L-14-336'   # Meta Perception Encoder L/14, 1024d, 336px; G14-448 сильнее но тяжелее (EMB_DIM=1280), B16-224 легче
BATCH_SIZE = 64                               # 336px - подними, если хватает памяти
NUM_WORKERS = 4
EMB_DIM = 1024                                # обнови под вариант модели
INPUT_DIR = Path('/kaggle/input')
WORK_DIR = Path('/kaggle/working')
IMAGES_ROOT = '/kaggle/input/avito-film-camera-body-type/image'   # None = искать автоматически
OUT_EMB = WORK_DIR / 'image_emb_pe.parquet'
OUT_SIM = WORK_DIR / 'text_sim_pe.parquet'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EXPECTED_ROWS = 144313

# промпты из error-analysis (типы камер + should_abstain ловушки); порядок = порядок колонок sim_*
PROMPTS = {
    'slr': 'a photo of a single-lens reflex film camera',
    'tlr': 'a photo of a twin-lens reflex camera',
    'rangefinder': 'a photo of a rangefinder film camera',
    'compact': 'a photo of a compact point-and-shoot film camera',
    'instant': 'a photo of an instant camera',
    'accessory': 'a photo of a camera accessory such as a lens, viewfinder or flash',
    'lot': 'a photo of several cameras together, a lot or a bundle',
    'parts': 'a photo of broken camera parts for repair or spares',
    'box': 'a photo of a camera box or packaging without the camera',
    'digital': 'a photo of a modern digital camera',
    'movie': 'a photo of a movie or cine film camera',
    'film_roll': 'a photo of rolls of photographic film',
    'camera': 'a photo of a film camera',
    'not_camera': 'a photo of an object that is not a camera',
}

## Шаг 1: найти фото

In [ ]:
def find_images_root():
    """вернуть Path к папке с фото: заданную руками, готовый датасет или распакованный zip"""
    if IMAGES_ROOT is not None and Path(IMAGES_ROOT).exists():
        return Path(IMAGES_ROOT)
    if next(INPUT_DIR.rglob('*.jpg'), None) is not None:
        return INPUT_DIR
    zip_path = next(INPUT_DIR.rglob('*.zip'))
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(WORK_DIR / 'images')
    return WORK_DIR / 'images'

In [ ]:
images_root = find_images_root()
print('фото лежат в', images_root)

## Шаг 2: список путей

In [ ]:
def list_image_paths(root):
    """все пути к jpg отсортированные по image_id"""
    paths = list(Path(root).rglob('*.jpg'))
    return sorted(paths, key=lambda p: int(p.stem))

In [ ]:
paths = list_image_paths(images_root)
print('найдено фото:', len(paths))

## Шаг 3: модель, препроцесс, датасет

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model = model.eval().to(DEVICE)
print('загружена', MODEL_NAME, '| context_length:', model.context_length)

In [ ]:
class ImageFiles(Dataset):
    """отдаёт image_id и предобработанный тензор картинки (трансформ open_clip)"""

    def __init__(self, paths, preprocess):
        self.paths = paths
        self.preprocess = preprocess

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        image = Image.open(self.paths[i]).convert('RGB')
        return int(self.paths[i].stem), self.preprocess(image)


def build_loader(paths):
    """собрать даталоадер по списку путей"""
    return DataLoader(ImageFiles(paths, preprocess), batch_size=BATCH_SIZE,
                      num_workers=NUM_WORKERS, pin_memory=True)

## Шаг 4: энкодер + текстовые фичи промптов

In [ ]:
class ImageEncoder(torch.nn.Module):
    """forward = encode_image, чтобы DataParallel реально делил батч по GPU"""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pixels):
        return self.model.encode_image(pixels, normalize=False)


def setup_encoder(model):
    """L2-нормированные текст-фичи промптов (один forward) + image-энкодер (DataParallel при >1 GPU)"""
    tokens = tokenizer(list(PROMPTS.values()), context_length=model.context_length).to(DEVICE)
    with torch.no_grad():
        text_norm = model.encode_text(tokens, normalize=True).float()
    encoder = ImageEncoder(model)
    if torch.cuda.device_count() > 1:
        encoder = torch.nn.DataParallel(encoder)
    return encoder, text_norm

In [ ]:
def embed_batch(encoder, text_norm, pixels):
    """вернуть (сырой emb float32 numpy, косинусы к промптам float32 numpy) для батча"""
    pixels = pixels.to(DEVICE, non_blocking=True)
    with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
        feat = encoder(pixels)
    feat = feat.float()
    raw = feat.cpu().numpy()
    sims = (F.normalize(feat, dim=-1) @ text_norm.T).cpu().numpy()
    return raw, sims

## Шаг 5: прогон

In [ ]:
@torch.no_grad()
def extract_all(encoder, text_norm, loader):
    """прогнать все батчи: image_id, сырые эмбеддинги, косинусы к промптам"""
    ids, embs, sims = [], [], []
    for batch_ids, pixels in tqdm(loader):
        raw, sim = embed_batch(encoder, text_norm, pixels)
        embs.append(raw)
        sims.append(sim)
        ids.append(batch_ids.numpy())
    return np.concatenate(ids), np.concatenate(embs), np.concatenate(sims)

In [ ]:
encoder, text_norm = setup_encoder(model)
ids, embs, sims = extract_all(encoder, text_norm, build_loader(paths))
print('эмбеддинги:', embs.shape, '| косинусы:', sims.shape)

## Шаг 6: собрать и сохранить (два parquet)

In [ ]:
def to_emb_dataframe(ids, embs):
    """image_id + сырой эмбеддинг emb_0..emb_{EMB_DIM-1}"""
    df = pd.DataFrame(embs.astype('float32'), columns=[f'emb_{i}' for i in range(EMB_DIM)])
    df.insert(0, 'image_id', ids.astype('int64'))
    return df


def to_sim_dataframe(ids, sims):
    """image_id + косинусы к промптам sim_<name> (порядок PROMPTS)"""
    df = pd.DataFrame(sims.astype('float32'), columns=[f'sim_{name}' for name in PROMPTS])
    df.insert(0, 'image_id', ids.astype('int64'))
    return df

In [ ]:
emb_df = to_emb_dataframe(ids, embs)
sim_df = to_sim_dataframe(ids, sims)
emb_df.to_parquet(OUT_EMB, index=False)
sim_df.to_parquet(OUT_SIM, index=False)
print('сохранено:', OUT_EMB, '|', OUT_SIM)

## Шаг 7: проверки

In [ ]:
for name, df, width in [('emb', emb_df, EMB_DIM), ('sim', sim_df, len(PROMPTS))]:
    assert len(df) == EXPECTED_ROWS, f'{name}: строк {len(df)} вместо {EXPECTED_ROWS}'
    assert df['image_id'].nunique() == EXPECTED_ROWS, f'{name}: image_id не уникален'
    assert df.isna().sum().sum() == 0, f'{name}: есть NaN'
    assert df.shape[1] == width + 1, f'{name}: неверное число столбцов'
print('emb:', emb_df.shape, '| sim:', sim_df.shape)
print('промпты:', list(PROMPTS))
print('OK')

## Что скачать

Скачай ОБА parquet из панели Output справа и положи локально в training/artifacts/embeddings/:
- image_emb_pe.parquet
- text_sim_pe.parquet

Схема (контракт для обвязки features.py, следующий шаг):
- image_emb_pe.parquet: image_id:int64 + emb_0..emb_1023:float32 (СЫРОЙ, L2/standard наложит пайплайн)
- text_sim_pe.parquet: image_id:int64 + sim_<name>:float32 (косинус, 14 колонок), оба keyed image_id

Дальше: source-aware рефактор features.py (мультиисточник image_emb: PE рядом с DINOv3) + эксперимент champion+pe vs champion (matched-frontier на audit)